# CPIC on Particle Orbit Data Generation

In [ ]:
import numpy as np
import torch

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import sys
from pathlib import Path
print(sys.executable)

# Paths: experiment (for generate_particle_orbit), and src (for cpic package)
experiment_path = Path("../experiments/particle_orbit_experiment").resolve()
src_path = Path("../src").resolve()
for p in (experiment_path, src_path):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from generate_particle_orbit import (
    generate_particle_orbit_positions,
    generate_particle_orbit_process_timeseries,
    animate_particle_orbit_process,
    plot_verification_3d)
from filter_visualization import (
    plot_filter_heatmap_panels,
    plot_physical_filter_heatmaps,
    compute_orbit_in_grid_coords,
    compute_avg_density_grid,
    compute_tangential_arrows,
    plot_orbit_density_map,
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from cpic import CPIC
from cpic.utils.data import PastFutureDataset

# Simulate the Particle Orbit Data Generation Process

In [ ]:
num_noise = 80 # modify depending on how much noise you want

In [ ]:
ani = animate_particle_orbit_process(t_max=200, 
                                      num_blob=30, 
                                      num_noise=num_noise, 
                                      orbit_radius=3.0, 
                                      omega=0.05, 
                                      sigma_blob=0.1, 
                                      noise_ar_coeff=0.8,
                                      spatial_bounds=10.0,
                                      seed=42, 
                                      interval=50)
plt.close(ani._fig)
HTML(ani.to_jshtml())

In [ ]:
positions = generate_particle_orbit_positions(t_max=2000,
                                               num_blob=30,
                                               num_noise=num_noise,
                                               orbit_radius=3.0,
                                               omega=0.05,
                                               sigma_blob=0.1,
                                               noise_ar_coeff=0.8,
                                               spatial_bounds=10.0,
                                               seed=42)
fig, ax = plot_verification_3d(positions, num_blob=30, num_noise=num_noise)
plt.show()

# Generate the Data (interleaved particle timeseries)

Observations are shape `(t_max, N*2)`: particles sorted by **x at t=0**, columns `[x0,y0,x1,y1,...]`, z-scored per column. CPIC uses this as `xdim = N*2`.


In [ ]:
data, gt_latent, particle_order, particle_labels, standardization_mean, standardization_std, positions = generate_particle_orbit_process_timeseries(
    t_max=2000,
    num_blob=30,
    num_noise=num_noise,
    orbit_radius=3.0,
    omega=0.05,
    sigma_blob=0.1,
    noise_ar_coeff=0.8,
    spatial_bounds=10.0,
    seed=42,
)
n_spatial = data.shape[1]
print("data shape", data.shape)
print("particle_order (x-sort at t=0):", particle_order.shape, "labels (blob=1/noise=0, orig order):", particle_labels.shape)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(data.T, aspect="auto")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Feature index (interleaved x,y; x-sorted)")
axes[0].set_title("Features x time\n(particle timeseries)")

axes[1].imshow(data, aspect="auto")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("Time")
axes[1].set_title("Time x features")

axes[2].plot(gt_latent[:, 0], gt_latent[:, 1], "b--", alpha=0.7)
axes[2].set_xlabel("cos($\omega$ t)")
axes[2].set_ylabel("sin($\omega$ t)")
axes[2].set_aspect("equal")
axes[2].set_title("GT circular latent motion")

plt.tight_layout()
plt.show()


In [ ]:
def animate_particle_timeseries(data, interval=80):
    T, F = data.shape
    fig, ax = plt.subplots(figsize=(12, 2.5))
    lo, hi = float(data.min()), float(data.max())
    im = ax.imshow(data[0:1], aspect="auto", vmin=lo, vmax=hi, cmap="viridis")
    ax.set_xlabel("Feature index (x0,y0,x1,y1,... after x-sort at t=0)")
    ax.set_ylabel("frame")
    plt.colorbar(im, ax=ax)
    def update(frame):
        im.set_data(data[frame : frame + 1])
        ax.set_title(f"t = {frame}")
        return (im,)
    return animation.FuncAnimation(fig, update, frames=T, interval=interval, blit=False)

anim1 = animate_particle_timeseries(data)
plt.close(anim1._fig)
HTML(anim1.to_jshtml())


# Run CPIC

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "mps"
print("device:", device)

In [ ]:
T = 10
period_steps = int(round((2 * np.pi) / 0.05))

# Train/test split on time to avoid probing on the same windows used for CPIC training.
t_split = int(0.7 * len(data))
if t_split <= 2*T or (len(data) - t_split) <= 2*T:
    raise ValueError("t_split must leave enough room for train/test past-future windows.")

train_data = PastFutureDataset([data[:t_split]], window_size=T)
print(f"total timesteps={len(data)}, train={t_split}, test={len(data) - t_split}")
print(f"num train past/future windows for CPIC fit: {len(train_data)}")

### Helper functions 

In [ ]:
def build_past_windows_and_gt(data, gt_latent, window_size, model, t_min=None, t_max=None):
    """Build past windows in [t_min, t_max) and align GT to each window end time."""
    n_timesteps = len(data)
    t_min = window_size if t_min is None else max(window_size, int(t_min))
    t_max = n_timesteps if t_max is None else min(n_timesteps, int(t_max))
    if t_max <= t_min:
        raise ValueError(f"Invalid time span: t_min={t_min}, t_max={t_max}")

    end_times = np.arange(t_min, t_max)
    past_windows = np.stack([data[t - window_size : t] for t in end_times], axis=0)
    past_tensor = torch.from_numpy(past_windows).float().to(device)

    with torch.no_grad():
        encoded_windows = model.encode(past_tensor) # (num_windows, T, ydim)
        z = encoded_windows[:, -1, :].cpu().numpy() # window-end latent (num_windows, ydim)

    gt = gt_latent[end_times - 1]
    return z, gt


def evaluate_probe_train_test(z_train, gt_train, z_test, gt_test, encoder_name="mlp", period_steps=None):
    """Fit probe on train latents; plot train and held-out test trajectories side by side."""
    reg = LinearRegression().fit(z_train, gt_train)
    pred_train = reg.predict(z_train)
    pred_test = reg.predict(z_test)

    r2_train_x = r2_score(gt_train[:, 0], pred_train[:, 0])
    r2_train_y = r2_score(gt_train[:, 1], pred_train[:, 1])
    r2_train = r2_score(gt_train, pred_train)

    r2_test_x = r2_score(gt_test[:, 0], pred_test[:, 0])
    r2_test_y = r2_score(gt_test[:, 1], pred_test[:, 1])
    r2_test = r2_score(gt_test, pred_test)

    print(f"{encoder_name} - train R^2: x={r2_train_x:.2f}, y={r2_train_y:.2f} | "
          f"test R^2: x={r2_test_x:.2f}, y={r2_test_y:.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Train: GT vs probe prediction (same regressor, evaluated on train z)
    ax = axes[0]
    ax.plot(gt_train[:, 0], gt_train[:, 1], "b--", label="Ground truth", linewidth=2)
    ax.plot(pred_train[:, 0], pred_train[:, 1], "r-", label=f"Predicted (train $R^2$={r2_train:.2f})")
    if period_steps is not None and period_steps > 0:
        idx_tr = np.arange(len(gt_train))
        mask_tr = (idx_tr % int(period_steps)) == 0
        ax.scatter(gt_train[mask_tr, 0], gt_train[mask_tr, 1], c="navy", s=20, marker="x", label="GT period marks")
        ax.scatter(pred_train[mask_tr, 0], pred_train[mask_tr, 1], c="darkred", s=20, marker="x", label="Pred period marks")
    ax.set_title(f"Training set (first {len(gt_train)} windows)\n{encoder_name} encoder")
    ax.axis("equal")
    ax.legend(loc="upper right", fontsize=8)

    # Test: same regressor, held-out z
    ax = axes[1]
    ax.plot(gt_test[:, 0], gt_test[:, 1], "b--", label="Ground truth", linewidth=2)
    ax.plot(pred_test[:, 0], pred_test[:, 1], "r-", label=f"Predicted (test $R^2$={r2_test:.2f})")
    if period_steps is not None and period_steps > 0:
        idx_te = np.arange(len(gt_test))
        period_mask = (idx_te % int(period_steps)) == 0
        ax.scatter(gt_test[period_mask, 0], gt_test[period_mask, 1], c="navy", s=20, marker="x", label="GT period marks")
        ax.scatter(pred_test[period_mask, 0], pred_test[period_mask, 1], c="darkred", s=20, marker="x", label="Pred period marks")
    ax.set_title(f"Test set (held-out) (last {len(gt_test)} windows)\n{encoder_name} encoder")
    ax.axis("equal")
    ax.legend(loc="upper right", fontsize=8)
    
    fig.suptitle("CPIC linear probe (fit on train latents; test uses same regressor)", y=1.02)
    fig.tight_layout()
    plt.show()

    return {"r2_train": r2_train, "r2_train_x": r2_train_x, "r2_train_y": r2_train_y, 
            "r2_test": r2_test, "r2_test_x": r2_test_x, "r2_test_y": r2_test_y
            }


def estimate_feature_pi_scores(data, t_split, *, max_lag=5, lag_agg="mean"):
    """Feature-wise PI proxy from lagged autocorrelation on the train split."""
    train = data[:t_split]
    max_lag = max(1, int(max_lag))
    if lag_agg not in {"mean", "max"}:
        raise ValueError(f"lag_agg must be 'mean' or 'max', got {lag_agg!r}")
    if train.shape[0] < max_lag + 2:
        return np.ones(train.shape[1], dtype=np.float32)

    lag_scores = []
    eps = 1e-8
    for lag in range(1, max_lag + 1):
        x_prev = train[:-lag]
        x_next = train[lag:]
        scores = np.zeros(train.shape[1], dtype=np.float32)
        for feat in range(train.shape[1]):
            a = x_prev[:, feat]
            b = x_next[:, feat]
            if np.std(a) < eps or np.std(b) < eps:
                scores[feat] = 0.0
                continue
            corr = np.corrcoef(a, b)[0, 1]
            scores[feat] = 0.0 if np.isnan(corr) else abs(float(corr))
        lag_scores.append(scores)

    lag_scores_arr = np.stack(lag_scores, axis=0)
    scores = np.max(lag_scores_arr, axis=0) if lag_agg == "max" else np.mean(lag_scores_arr, axis=0)
    max_val = float(np.max(scores))
    return np.ones_like(scores, dtype=np.float32) if max_val <= eps else (scores / max_val).astype(np.float32)


def with_mask_init_from_pi(encoder_params, pi_scores):
    params = dict(encoder_params)
    if params.get("encoder_type") != "mask_mlp":
        return params
    if params.get("mask_strategy") in {"pi_static", "pi_init_learned"}:
        params["mask_init_values"] = pi_scores
    return params


def get_final_mask_stats(model, threshold=0.5):
    if getattr(model.encoder, "encoder_type", None) != "mask_mlp":
        return float("nan"), ""
    mask_tensor = model.encoder.get_feature_mask().detach().cpu().numpy()
    active = np.where(mask_tensor >= threshold)[0]
    return float(active.size / mask_tensor.size), ",".join(map(str, active.tolist()))


def plot_feature_mask(
    model,
    *,
    pi_scores=None,
    particle_labels=None,
    particle_order=None,
    threshold=0.5,
    title=None,
):
    """Bar plot of learned mask vs optional PI initialization scores."""
    enc = model.encoder
    if getattr(enc, "encoder_type", None) != "mask_mlp":
        print("Skipping mask plot: encoder is not mask_mlp")
        return

    mask = enc.get_feature_mask().detach().cpu().numpy()
    probs = torch.sigmoid(enc._mean.mask_logits).detach().cpu().numpy()
    n_feat = mask.size
    x = np.arange(n_feat)

    fig, ax = plt.subplots(figsize=(12, 3.5))
    colors = None
    if particle_labels is not None and particle_order is not None:
        sorted_labels = particle_labels[particle_order]
        feat_labels = np.repeat(sorted_labels, 2)
        colors = np.where(feat_labels == 1, "tab:blue", "tab:orange")

    ax.bar(x, probs, color=colors, alpha=0.85, label="mask prob")
    ax.axhline(threshold, color="k", linestyle="--", linewidth=1, label=f"threshold={threshold}")
    if pi_scores is not None:
        ax.plot(x, pi_scores, "g--", linewidth=1.5, alpha=0.8, label="PI init scores")
    active_frac, active_idx = get_final_mask_stats(model, threshold)
    ax.set_xlabel("Feature index (2 per sorted particle: x, y)")
    ax.set_ylabel("Gate probability")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="upper right", fontsize=8)
    subtitle = "active={:.2f} | indices={}".format(active_frac, active_idx[:80] + ("..." if len(active_idx) > 80 else ""))
    ax.set_title((title or "Feature mask") + "\n" + subtitle)
    plt.tight_layout()
    plt.show()
    return fig, ax


In [ ]:
import gc

def cleanup_torch(verbose=True):
    """Release Python refs and ask torch backends to free cached memory."""
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        if verbose:
            alloc = torch.cuda.memory_allocated() / (1024 ** 2)
            reserved = torch.cuda.memory_reserved() / (1024 ** 2)
            print(f"[cleanup] CUDA allocated={alloc:.1f} MB, reserved={reserved:.1f} MB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        if verbose:
            print("[cleanup] MPS backend active (no torch empty_cache equivalent).")
    elif verbose:
        print("[cleanup] CPU backend active.")


def cleanup_model_resources(*objs, verbose=True):
    """Drop references to large training objects, then clear backend caches."""
    for obj in objs:
        del obj
    cleanup_torch(verbose=verbose)

## MLP encoder

In [ ]:
encoder_params = {
    "deterministic": False,
    "encoder_type": "mlp",
    "linear_encoder": False,
    "n_layers": 1,
    "activation": "relu",
}

In [ ]:
cpic = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params, hidden_dim=64, beta=1e-5, device=device)
cpic.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)

In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic, t_min=t_split, t_max=len(data))

mlp_metrics = evaluate_probe_train_test(
    z_train,
    gt_train,
    z_test,
    gt_test,
    encoder_name="MLP",
    period_steps=period_steps,
)

In [ ]:
cleanup_model_resources(cpic, loss, I_compress, I_predictive)

## ConvSpatial (Latent predictive space) encoder

`conv_spatial` applies Conv2d along the **feature** axis (particle channels), suitable for the `(t_max, N*2)` interleaved representation.


In [ ]:
encoder_params = {
    "deterministic": False,
    "encoder_type": "conv_spatial",
    "linear_encoder": False,
    "n_layers": 1,
    "activation": "relu",
    "conv_kernel_size": 5,
    "conv_stride": 1,
    "conv_padding": 2,
}


In [ ]:
cpic_conv_latent = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params, hidden_dim=64, beta=1e-5, device=device)
cpic_conv_latent.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic_conv_latent.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_latent, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_latent, t_min=t_split, t_max=len(data))

conv_latent_metrics = evaluate_probe_train_test(
    z_train,
    gt_train,
    z_test,
    gt_test,
    encoder_name="ConvSpatial (L)",
    period_steps=period_steps,
)

### Kernel / feature analysis

The kernels slide along the **feature** axis (interleaved particle coordinates).


In [ ]:
enc = cpic_conv_latent.encoder
layer_idx = 0

if getattr(enc, "encoder_type", None) != "conv_spatial":
    print("Skipping kernel analysis: encoder is not conv_spatial")
else:
    w, meta = enc.get_filters(layer_idx=layer_idx)
    K_h, K_w = meta["kernel_size"]
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "padding:", meta["padding"], "stride:", meta["stride"])

In [ ]:
w_mag = np.abs(w[:, 0, :, 0]) # (C_out, K_h)
fig, axes = plot_filter_heatmap_panels(w_mag, gap_rows=1, filters_per_panel=8, suptitle=f"conv_spatial mean layer {layer_idx}")
plt.show()

In [ ]:
cleanup_model_resources(cpic_conv_latent, loss, I_compress, I_predictive)

## ConvSpatial (Observation predictive space) encoder

In [ ]:
cpic_conv_obs = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params, hidden_dim=64, beta=1e-5, device=device, predictive_space="observation")
cpic_conv_obs.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic_conv_obs.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_obs, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_obs, t_min=t_split, t_max=len(data))

conv_obs_metrics = evaluate_probe_train_test(
    z_train,
    gt_train,
    z_test,
    gt_test,
    encoder_name="ConvSpatial (O)",
    period_steps=period_steps,
)

### Kernel / feature analysis

In [ ]:
enc_obs = cpic_conv_obs.encoder
layer_idx = 0

if getattr(enc_obs, "encoder_type", None) != "conv_spatial":
    print("Skipping kernel analysis: encoder is not conv_spatial")
else:
    w, meta = enc_obs.get_filters(layer_idx=layer_idx)
    K_h, K_w = meta["kernel_size"]
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "padding:", meta["padding"], "stride:", meta["stride"])

In [ ]:
w_mag = np.abs(w[:, 0, :, 0]) # (C_out, K_h)
fig, axes = plot_filter_heatmap_panels(w_mag, gap_rows=1, filters_per_panel=8, suptitle=f"conv_spatial mean layer {layer_idx}")
plt.show()

In [ ]:
cleanup_model_resources(cpic_conv_obs, loss, I_compress, I_predictive)

## ConvParticle2D (Observation predictive space) encoder

`conv_particle2d` reshapes each timestep into a `(num_particles, 2)` tensor (x,y) and applies Conv2d over the particle axis (and optionally across x/y together). This makes the learned kernels interpretable as local patterns over neighboring particles in the x-sorted order at `t=0`.


In [ ]:
encoder_params_particle2d = {
    "deterministic": False,
    "encoder_type": "conv_particle",
    "linear_encoder": False,
    "n_layers": 1,
    "activation": "relu",
    # conv over particle axis
    "conv_kernel_size": 5,
    "conv_stride": 1,
    "conv_padding": 2,
    # conv over coordinate axis (1 = separate x/y, 2 = mix x/y)
    "conv_coord_kernel_size": 2,
}

cpic_particle2d_obs = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params_particle2d, hidden_dim=64, beta=1e-5, device=device, predictive_space="observation")
cpic_particle2d_obs.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic_particle2d_obs.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)

In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_particle2d_obs, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_particle2d_obs, t_min=t_split, t_max=len(data))

particle2d_metrics = evaluate_probe_train_test(
    z_train,
    gt_train,
    z_test,
    gt_test,
    encoder_name="ConvSpatialParticle2D (O)",
    period_steps=period_steps,
)

### Kernel / particle analysis (ConvParticle2D)

We visualize the first conv layer’s weights as \(|w|\) and compute a simple per-particle “importance” score by distributing kernel magnitudes back to the particle indices each output position “sees,” analogous to the `conv_spatial` feature-importance approach.


In [ ]:
enc_p2d = cpic_particle2d_obs.encoder
layer_idx = 0

if getattr(enc_p2d, "encoder_type", None) != "conv_particle2d":
    print("Skipping kernel analysis: encoder is not conv_particle2d")
else:
    w, meta = enc_p2d.get_filters(layer_idx=layer_idx)
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "padding:", meta["padding"], "stride:", meta["stride"], "kernel:", meta["kernel_size"])

In [ ]:
# First-layer weights: (C_out, 1, K_particles, K_coords)
w_mag = np.abs(w[:, 0, :, :])
C_out, Kp, Kc = w_mag.shape

# Heatmap panel: collapse coord dim so we can reuse existing panel plotting helper
w_mag_1d = w_mag.sum(axis=2)  # (C_out, K_particles)
fig, axes = plot_filter_heatmap_panels(
    w_mag_1d,
    gap_rows=1,
    filters_per_panel=8,
    suptitle=f"conv_particle2d mean layer {layer_idx} (sum over coord dim, Kc={Kc})",
)
plt.show()

In [ ]:
cleanup_model_resources(cpic_particle2d_obs, loss, I_compress, I_predictive)

## ConvParticle2D (Latent predictive space) encoder

Same encoder as above, but trains CPIC with `predictive_space="latent"` (predicting encoded future latents rather than raw future observations).


In [ ]:
cpic_particle2d_latent = CPIC(ydim=2, xdim=n_spatial, T=T, encoder_params=encoder_params_particle2d, hidden_dim=64, beta=1e-5, device=device, predictive_space="latent")
cpic_particle2d_latent.to(device);

In [ ]:
loss, I_compress, I_predictive = cpic_particle2d_latent.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)

In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_particle2d_latent, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_particle2d_latent, t_min=t_split, t_max=len(data))

particle2d_latent_metrics = evaluate_probe_train_test(
    z_train,
    gt_train,
    z_test,
    gt_test,
    encoder_name="ConvParticle2D (L)",
    period_steps=period_steps,
)

### Kernel / particle analysis

In [ ]:
enc_p2d_latent = cpic_particle2d_latent.encoder
layer_idx = 0

if getattr(enc_p2d_latent, "encoder_type", None) != "conv_particle2d":
    print("Skipping kernel analysis: encoder is not conv_particle2d")
else:
    w, meta = enc_p2d_latent.get_filters(layer_idx=layer_idx)
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "padding:", meta["padding"], "stride:", meta["stride"], "kernel:", meta["kernel_size"])

In [ ]:
w_mag = np.abs(w[:, 0, :, :])
C_out, Kp, Kc = w_mag.shape
w_mag_1d = w_mag.sum(axis=2)
fig, axes = plot_filter_heatmap_panels(
    w_mag_1d,
    gap_rows=1,
    filters_per_panel=8,
    suptitle=f"conv_particle2d mean layer {layer_idx} (latent objective; sum coord dim, Kc={Kc})",
)
plt.show()

In [ ]:
cleanup_model_resources(cpic_particle2d_latent, loss, I_compress, I_predictive)

## Feature masking encoders (`mask_mlp`)

`mask_mlp` applies a per-feature Bernoulli gate (straight-through) before an MLP projection.
Three initialization strategies mirror the batch experiment config:

- **MaskRandom**: fixed random logits (`mask_learnable=False`, `mask_init="random"`)
- **MaskPIStatic**: PI-derived scores frozen at init (`mask_strategy="pi_static"`)
- **MaskPIInitLearned**: PI-derived init, mask logits learned during training (`mask_strategy="pi_init_learned"`)

PI scores are estimated from lagged autocorrelation on the train split (same as `run_particle_orbit_experiment.py`).


In [ ]:
pi_max_lag = 5
pi_lag_agg = "mean"
mask_eval_threshold = 0.6

pi_scores = estimate_feature_pi_scores(data, t_split, max_lag=pi_max_lag, lag_agg=pi_lag_agg)
print(f"PI scores: min={pi_scores.min():.3f}, max={pi_scores.max():.3f}, mean={pi_scores.mean():.3f}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(pi_scores, color="green", linewidth=1)
ax.set_title("Feature-wise PI proxy scores (train split)")
ax.set_xlabel("Feature index")
ax.set_ylabel("Normalized score")
plt.tight_layout()
plt.show()


## MaskRandom (Latent) encoder

In [ ]:
encoder_params_maskrandom_latent = {   'activation': 'relu',
    'deterministic': False,
    'encoder_type': 'mask_mlp',
    'linear_encoder': False,
    'mask_init': 'random',
    'mask_learnable': False,
    'mask_strategy': 'random',
    'n_layers': 1}

cpic_maskrandom_latent = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_maskrandom_latent,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="latent",
)
cpic_maskrandom_latent.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_maskrandom_latent.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_maskrandom_latent, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_maskrandom_latent, t_min=t_split, t_max=len(data))

metrics_maskrandom_latent = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="MaskRandom (Latent)",
    period_steps=period_steps,
)


### Mask analysis

In [ ]:
plot_feature_mask(
    cpic_maskrandom_latent,
        particle_labels=particle_labels,
    particle_order=particle_order,
    threshold=mask_eval_threshold,
    title="MaskRandom (Latent) — feature gates (blue=blob, orange=noise)",
)
active_frac, active_idx = get_final_mask_stats(cpic_maskrandom_latent, mask_eval_threshold)
print(f"active_frac={active_frac:.3f}, active_indices={active_idx}")


In [ ]:
cleanup_model_resources(cpic_maskrandom_latent, loss, I_compress, I_predictive)


## MaskRandom (Observation) encoder

In [ ]:
encoder_params_maskrandom_observation = {   'activation': 'relu',
    'deterministic': False,
    'encoder_type': 'mask_mlp',
    'linear_encoder': False,
    'mask_init': 'random',
    'mask_learnable': False,
    'mask_strategy': 'random',
    'n_layers': 1}

cpic_maskrandom_observation = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_maskrandom_observation,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="observation",
)
cpic_maskrandom_observation.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_maskrandom_observation.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_maskrandom_observation, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_maskrandom_observation, t_min=t_split, t_max=len(data))

metrics_maskrandom_observation = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="MaskRandom (Observation)",
    period_steps=period_steps,
)


### Mask analysis

In [ ]:
plot_feature_mask(
    cpic_maskrandom_observation,
        particle_labels=particle_labels,
    particle_order=particle_order,
    threshold=mask_eval_threshold,
    title="MaskRandom (Observation) — feature gates (blue=blob, orange=noise)",
)
active_frac, active_idx = get_final_mask_stats(cpic_maskrandom_observation, mask_eval_threshold)
print(f"active_frac={active_frac:.3f}, active_indices={active_idx}")


In [ ]:
cleanup_model_resources(cpic_maskrandom_observation, loss, I_compress, I_predictive)


## MaskPIStatic (Latent) encoder

In [ ]:
encoder_params_maskpistatic_latent = with_mask_init_from_pi(
{   'activation': 'relu',
    'deterministic': False,
    'encoder_type': 'mask_mlp',
    'linear_encoder': False,
    'mask_init': 'pi',
    'mask_learnable': False,
    'mask_strategy': 'pi_static',
    'n_layers': 1},
    pi_scores,
)

cpic_maskpistatic_latent = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_maskpistatic_latent,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="latent",
)
cpic_maskpistatic_latent.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_maskpistatic_latent.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpistatic_latent, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpistatic_latent, t_min=t_split, t_max=len(data))

metrics_maskpistatic_latent = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="MaskPIStatic (Latent)",
    period_steps=period_steps,
)


### Mask analysis

In [ ]:
plot_feature_mask(
    cpic_maskpistatic_latent,
    pi_scores=pi_scores,
        particle_labels=particle_labels,
    particle_order=particle_order,
    threshold=mask_eval_threshold,
    title="MaskPIStatic (Latent) — feature gates (blue=blob, orange=noise)",
)
active_frac, active_idx = get_final_mask_stats(cpic_maskpistatic_latent, mask_eval_threshold)
print(f"active_frac={active_frac:.3f}, active_indices={active_idx}")


In [ ]:
cleanup_model_resources(cpic_maskpistatic_latent, loss, I_compress, I_predictive)


## MaskPIStatic (Observation) encoder

In [ ]:
encoder_params_maskpistatic_observation = with_mask_init_from_pi(
{   'activation': 'relu',
    'deterministic': False,
    'encoder_type': 'mask_mlp',
    'linear_encoder': False,
    'mask_init': 'pi',
    'mask_learnable': False,
    'mask_strategy': 'pi_static',
    'n_layers': 1},
    pi_scores,
)

cpic_maskpistatic_observation = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_maskpistatic_observation,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="observation",
)
cpic_maskpistatic_observation.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_maskpistatic_observation.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpistatic_observation, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpistatic_observation, t_min=t_split, t_max=len(data))

metrics_maskpistatic_observation = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="MaskPIStatic (Observation)",
    period_steps=period_steps,
)


### Mask analysis

In [ ]:
plot_feature_mask(
    cpic_maskpistatic_observation,
    pi_scores=pi_scores,
        particle_labels=particle_labels,
    particle_order=particle_order,
    threshold=mask_eval_threshold,
    title="MaskPIStatic (Observation) — feature gates (blue=blob, orange=noise)",
)
active_frac, active_idx = get_final_mask_stats(cpic_maskpistatic_observation, mask_eval_threshold)
print(f"active_frac={active_frac:.3f}, active_indices={active_idx}")


In [ ]:
cleanup_model_resources(cpic_maskpistatic_observation, loss, I_compress, I_predictive)


## MaskPIInitLearned (Latent) encoder

In [ ]:
encoder_params_maskpiinitlearned_latent = with_mask_init_from_pi(
{   'activation': 'relu',
    'deterministic': False,
    'encoder_type': 'mask_mlp',
    'linear_encoder': False,
    'mask_init': 'pi',
    'mask_learnable': True,
    'mask_strategy': 'pi_init_learned',
    'n_layers': 1},
    pi_scores,
)

cpic_maskpiinitlearned_latent = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_maskpiinitlearned_latent,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="latent",
)
cpic_maskpiinitlearned_latent.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_maskpiinitlearned_latent.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpiinitlearned_latent, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpiinitlearned_latent, t_min=t_split, t_max=len(data))

metrics_maskpiinitlearned_latent = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="MaskPIInitLearned (Latent)",
    period_steps=period_steps,
)


### Mask analysis

In [ ]:
plot_feature_mask(
    cpic_maskpiinitlearned_latent,
    pi_scores=pi_scores,
        particle_labels=particle_labels,
    particle_order=particle_order,
    threshold=mask_eval_threshold,
    title="MaskPIInitLearned (Latent) — feature gates (blue=blob, orange=noise)",
)
active_frac, active_idx = get_final_mask_stats(cpic_maskpiinitlearned_latent, mask_eval_threshold)
print(f"active_frac={active_frac:.3f}, active_indices={active_idx}")


In [ ]:
cleanup_model_resources(cpic_maskpiinitlearned_latent, loss, I_compress, I_predictive)


## MaskPIInitLearned (Observation) encoder

In [ ]:
encoder_params_maskpiinitlearned_observation = with_mask_init_from_pi(
{   'activation': 'relu',
    'deterministic': False,
    'encoder_type': 'mask_mlp',
    'linear_encoder': False,
    'mask_init': 'pi',
    'mask_learnable': True,
    'mask_strategy': 'pi_init_learned',
    'n_layers': 1},
    pi_scores,
)

cpic_maskpiinitlearned_observation = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_maskpiinitlearned_observation,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="observation",
)
cpic_maskpiinitlearned_observation.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_maskpiinitlearned_observation.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpiinitlearned_observation, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_maskpiinitlearned_observation, t_min=t_split, t_max=len(data))

metrics_maskpiinitlearned_observation = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="MaskPIInitLearned (Observation)",
    period_steps=period_steps,
)


### Mask analysis

In [ ]:
plot_feature_mask(
    cpic_maskpiinitlearned_observation,
    pi_scores=pi_scores,
        particle_labels=particle_labels,
    particle_order=particle_order,
    threshold=mask_eval_threshold,
    title="MaskPIInitLearned (Observation) — feature gates (blue=blob, orange=noise)",
)
active_frac, active_idx = get_final_mask_stats(cpic_maskpiinitlearned_observation, mask_eval_threshold)
print(f"active_frac={active_frac:.3f}, active_indices={active_idx}")


In [ ]:
cleanup_model_resources(cpic_maskpiinitlearned_observation, loss, I_compress, I_predictive)


## ConvPhysical (Latent predictive space) encoder

`conv_physical` bins standardized particle coordinates onto a 2D occupancy grid and applies symmetric Conv2d filters in physical space. Filters are directly interpretable as spatial detectors; we overlay the blob orbit on filter heatmaps and a time-averaged density map.


In [ ]:
conv_phys_base = {
    "deterministic": False,
    "encoder_type": "conv_physical",
    "linear_encoder": False,
    "n_layers": 0,
    "activation": "relu",
    "grid_size": 20,
    "spatial_bounds": 3.0,
    "conv_kernel_size": 3,
    "conv_stride": 1,
    "conv_padding": 1,
}
encoder_params_conv_phys = conv_phys_base.copy()

cpic_conv_phys_latent = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_conv_phys,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="latent",
)
cpic_conv_phys_latent.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_conv_phys_latent.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_phys_latent, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_phys_latent, t_min=t_split, t_max=len(data))

conv_phys_latent_metrics = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="ConvPhysical (L)",
    period_steps=period_steps,
)


### Physical filter + orbit analysis

In [ ]:
enc_phys = cpic_conv_phys_latent.encoder
layer_idx = 0

if getattr(enc_phys, "encoder_type", None) != "conv_physical":
    print("Skipping physical filter analysis: encoder is not conv_physical")
else:
    w, meta = enc_phys.get_filters(layer_idx=layer_idx)
    print(f"Layer {layer_idx} Conv2d weight shape:", w.shape, "grid_size:", meta["grid_size"], "spatial_bounds:", meta["spatial_bounds"])

    fig, axes = plot_physical_filter_heatmaps(
        w, n_cols=8, suptitle=f"conv_physical layer {layer_idx} (latent)",
    )
    plt.show()

    orbit_grid = compute_orbit_in_grid_coords(
        positions, particle_labels, particle_order,
        standardization_mean, standardization_std,
        grid_size=meta["grid_size"], spatial_bounds=meta["spatial_bounds"],
    )
    avg_density = compute_avg_density_grid(
        data[:t_split], grid_size=meta["grid_size"], spatial_bounds=meta["spatial_bounds"],
    )
    arrows = compute_tangential_arrows(orbit_grid)
    fig2, ax2 = plot_orbit_density_map(
        avg_density, orbit_grid, arrows=arrows,
        spatial_bounds=meta["spatial_bounds"],
        suptitle="ConvPhysical (L) — mean density + blob orbit",
    )
    plt.show()


In [ ]:
cleanup_model_resources(cpic_conv_phys_latent, loss, I_compress, I_predictive)


## ConvPhysical (Observation predictive space) encoder

In [ ]:
cpic_conv_phys_obs = CPIC(
    ydim=2, xdim=n_spatial, T=T,
    encoder_params=encoder_params_conv_phys,
    hidden_dim=64, beta=1e-5, device=device,
    predictive_space="observation",
)
cpic_conv_phys_obs.to(device);


In [ ]:
loss, I_compress, I_predictive = cpic_conv_phys_obs.fit(X=train_data, epochs=80, batch_size=256, lr=2e-4, early_stop=20)


In [ ]:
z_train, gt_train = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_phys_obs, t_min=T, t_max=t_split)
z_test, gt_test = build_past_windows_and_gt(data, gt_latent, T, cpic_conv_phys_obs, t_min=t_split, t_max=len(data))

conv_phys_obs_metrics = evaluate_probe_train_test(
    z_train, gt_train, z_test, gt_test,
    encoder_name="ConvPhysical (O)",
    period_steps=period_steps,
)


### Physical filter + orbit analysis

In [ ]:
enc_phys_obs = cpic_conv_phys_obs.encoder
layer_idx = 0

if getattr(enc_phys_obs, "encoder_type", None) != "conv_physical":
    print("Skipping physical filter analysis: encoder is not conv_physical")
else:
    w, meta = enc_phys_obs.get_filters(layer_idx=layer_idx)
    fig, axes = plot_physical_filter_heatmaps(
        w, n_cols=8, suptitle=f"conv_physical layer {layer_idx} (observation)",
    )
    plt.show()

    orbit_grid = compute_orbit_in_grid_coords(
        positions, particle_labels, particle_order,
        standardization_mean, standardization_std,
        grid_size=meta["grid_size"], spatial_bounds=meta["spatial_bounds"],
    )
    avg_density = compute_avg_density_grid(
        data[:t_split], grid_size=meta["grid_size"], spatial_bounds=meta["spatial_bounds"],
    )
    arrows = compute_tangential_arrows(orbit_grid)
    fig2, ax2 = plot_orbit_density_map(
        avg_density, orbit_grid, arrows=arrows,
        spatial_bounds=meta["spatial_bounds"],
        suptitle="ConvPhysical (O) — mean density + blob orbit",
    )
    plt.show()


In [ ]:
cleanup_model_resources(cpic_conv_phys_obs, loss, I_compress, I_predictive)
